# 25 · Capstone — Real-Time Fraud Detection

**Part 3 · Capstones** · *Prerequisites: everything* · *Milliseconds, adversaries, money*

The final boss combines every constraint the course has met: a decision **inside the
checkout's latency budget**, on **streaming features** that must be point-in-time honest,
for an event that is **1-in-300 rare**, labeled by chargebacks **30+ days late**, against
an adversary who **adapts** — and where every error has a price tag in actual dollars.

**Step 1 — Requirements.** Score every card transaction in **< 100 ms p99** (we are
*inside* the payment flow — an outage here is a revenue outage); decisions: allow /
step-up (review) / block; fraud base rate ~0.3%; analyst review capacity ≤ 5% of
transactions; chargebacks (ground truth) arrive ~30 days late; the system must degrade,
never die.

In [ ]:
from collections import defaultdict, deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=25)

**Step 2 — Architecture.** The request path touches only fast stores; everything heavy
hangs off the log:

In [ ]:
fig, ax = viz.system_diagram(
    nodes=[
        {"id": "tx", "x": 0, "y": 1, "label": "transaction", "kind": "client"},
        {"id": "api", "x": 2.1, "y": 1, "label": "decision\nservice"},
        {"id": "feat", "x": 4.3, "y": 2.1, "label": "online features\n(rolling aggregates)", "kind": "cache"},
        {"id": "model", "x": 4.3, "y": 1.0, "label": "model +\ncost policy", "kind": "service"},
        {"id": "fb", "x": 4.3, "y": -0.1, "label": "fallback rules", "kind": "note"},
        {"id": "log", "x": 2.1, "y": -1.2, "label": "event log", "kind": "queue"},
        {"id": "stream", "x": 4.6, "y": -2.1, "label": "stream consumer:\nfeature updates", "kind": "service"},
        {"id": "batch", "x": 7.0, "y": -1.2, "label": "point-in-time\ntraining builder +\nretrain (nb 20-21)", "kind": "service"},
        {"id": "review", "x": 7.0, "y": 1.0, "label": "analyst review\nqueue", "kind": "external"},
    ],
    edges=[
        {"src": "tx", "dst": "api", "label": "<100ms", "bidir": True},
        {"src": "api", "dst": "feat", "label": "get", "bidir": True},
        {"src": "api", "dst": "model", "bidir": True},
        {"src": "api", "dst": "fb", "label": "on timeout", "style": "dashed"},
        {"src": "api", "dst": "log", "label": "every event"},
        {"src": "log", "dst": "stream"},
        {"src": "stream", "dst": "feat", "label": "async update", "curve": -0.4},
        {"src": "log", "dst": "batch"},
        {"src": "api", "dst": "review", "label": "step-up cases", "style": "dashed"},
    ],
    title="Fast path reads; the log feeds everything else (nbs 11/12/16/21 in one box)",
)
plt.show()

## Step 3a — A world with fraud in it

Legitimate cards have habits (home country, typical amounts, human pacing). A stolen
card is a *burst*: rapid transactions, foreign country, escalating amounts. That
behavioral difference is what **velocity features** exist to capture:

In [ ]:
N_CARDS, N_DAYS = 2_000, 60
SECONDS = N_DAYS * 86_400
events = []

home_country = rng.integers(0, 12, N_CARDS)
typical_amount = rng.lognormal(3.2, 0.5, N_CARDS)          # ~$25 typical

for card in range(N_CARDS):
    # Legit traffic: ~1.2 transactions/day, human-paced.
    n = rng.poisson(1.2 * N_DAYS)
    for ts in np.sort(rng.uniform(0, SECONDS, n)):
        events.append({"ts": ts, "card": card, "fraud": 0,
                       "amount": float(rng.lognormal(np.log(typical_amount[card]), 0.6)),
                       "country": int(home_country[card] if rng.random() < 0.93
                                      else rng.integers(0, 12))})

fraud_cards = rng.choice(N_CARDS, size=60, replace=False)   # 3% of cards get hit
for card in fraud_cards:
    start = rng.uniform(0.2, 0.9) * SECONDS
    for k in range(int(rng.integers(4, 9))):                # the burst
        events.append({"ts": start + k * rng.uniform(30, 300), "card": card, "fraud": 1,
                       "amount": float(rng.lognormal(4.2 + 0.35 * k, 0.5)),   # escalating
                       "country": int(rng.integers(0, 12))})               # anywhere

events.sort(key=lambda e: e["ts"])
frame = pd.DataFrame(events)
print(f"{len(events):,} transactions, fraud rate {frame.fraud.mean():.2%}")
assert 0.002 < frame.fraud.mean() < 0.01, "properly rare"

## Step 3b — Streaming features, point-in-time honest

The **online feature store**: per-card rolling aggregates, updated by the stream
consumer as events flow. The same class serves two masters — live snapshots for the
request path, and the **training-set builder**, which replays history and snapshots each
transaction's features *strictly before* incorporating it. One definition, both worlds:
nb 16's feature-store discipline, now with money on it:

In [ ]:
class StreamFeatureStore:
    """Per-card state: recent timestamps/amounts/countries (bounded deques)."""

    def __init__(self):
        self.txns: dict[int, deque] = defaultdict(lambda: deque(maxlen=50))

    def snapshot(self, card: int, ts: float, amount: float) -> list[float]:
        """Features as of ts, BEFORE this transaction is applied."""
        past = [t for t in self.txns[card] if t["ts"] < ts]
        last_hour = [t for t in past if t["ts"] > ts - 3600]
        last_day = [t for t in past if t["ts"] > ts - 86_400]
        avg_amount = np.mean([t["amount"] for t in past]) if past else amount
        return [
            amount,
            amount / (avg_amount + 1e-9),                  # amount vs card's habit
            len(last_hour),                                # velocity, 1h
            len({t["country"] for t in last_day}),         # countries in 24h
            (ts - past[-1]["ts"]) if past else 86_400.0,   # seconds since previous
        ]

    def update(self, event: dict) -> None:
        self.txns[event["card"]].append(event)

FEATURES = ["amount", "amount_vs_habit", "txns_1h", "countries_24h", "secs_since_prev"]

# THE TRAINING BUILDER: replay the log; snapshot-then-update, event by event.
store = StreamFeatureStore()
X_rows, y_rows, ts_rows = [], [], []
for e in events:
    X_rows.append(store.snapshot(e["card"], e["ts"], e["amount"]))
    store.update(e)                                        # only NOW does it exist
X = np.array(X_rows)
y = frame.fraud.to_numpy()
ts_all = frame.ts.to_numpy()

# Time split (nb 16 §4.3): train on the first 40 days, evaluate on the rest.
split = ts_all < 40 * 86_400
model = HistGradientBoostingClassifier(class_weight="balanced", random_state=0) \
    .fit(X[split], y[split])
p_live = model.predict_proba(X[~split])[:, 1]
auc_honest = roc_auc_score(y[~split], p_live)
print(f"honest point-in-time features -> live AUC {auc_honest:.3f}")
assert auc_honest > 0.85, "velocity features see the burst as it happens"

**Failure first — the leak that ships itself.** The tempting shortcut: compute each
card's aggregates over its *whole history* with one groupby — one line instead of a
replay engine. A defrauded card's full-history stats *contain the fraud burst*, so the
features quietly borrow from the future. And here is why this bug ships: **offline, the
lazy builder looks just as good as the honest one.** Same AUC, a tenth of the code —
every incentive says use it. Then it meets production, where the future doesn't exist
yet:

In [ ]:
# The leaky builder: card-level aggregates over ALL 60 days, assigned to every
# row. Note especially 'txns_1h' becoming the card's LIFETIME PEAK hourly rate -
# for a defrauded card, that peak IS the fraud burst. The label, wearing a hat.
def peak_hourly_rate(ts_series) -> int:
    t = np.sort(ts_series.to_numpy())
    best, j = 1, 0
    for i in range(len(t)):
        while t[i] - t[j] > 3600:
            j += 1
        best = max(best, i - j + 1)
    return best

g = frame.groupby("card")
leaky = pd.DataFrame({
    "amount": frame.amount,
    "amount_vs_habit": frame.amount / frame.card.map(g.amount.mean()),
    "txns_1h": frame.card.map(g["ts"].apply(peak_hourly_rate)),   # lifetime peak!
    "countries_24h": frame.card.map(g.country.nunique()),         # lifetime countries!
    "secs_since_prev": frame.card.map(SECONDS / g.size()),        # avg gap
}).to_numpy()

leaky_model = HistGradientBoostingClassifier(class_weight="balanced", random_state=0) \
    .fit(leaky[split], y[split])
auc_leaky_offline = roc_auc_score(y[~split], leaky_model.predict_proba(leaky[~split])[:, 1])
# ...but at serving time only point-in-time features EXIST:
auc_leaky_served = roc_auc_score(y[~split], leaky_model.predict_proba(X[~split])[:, 1])

print(f"honest: {auc_honest:.3f} | leaky offline eval: {auc_leaky_offline:.3f} "
      f"(indistinguishable!) | leaky model served on real features: "
      f"{auc_leaky_served:.3f} (the crash)")
assert abs(auc_leaky_offline - auc_honest) < 0.03, \
    "offline, the shortcut looks FREE - identical metrics, far simpler code: the trap"
assert auc_leaky_served < auc_honest - 0.08, \
    "in production the time-traveling features don't exist; the honest model wins where it counts"

That triple of numbers is the single most expensive lesson in applied ML, and you have
now measured it: **the leaky model is indistinguishable offline — no metric review would
catch it — and it loses ~0.3 AUC the moment it serves.** The defense is structural, not
vigilance: one feature definition, replayed over history for training (this section),
computed live for serving — never two implementations (nb 16). No amount of offline
evaluation substitutes for that discipline, because offline is exactly where the bug is
invisible.

## Step 3c — From scores to decisions: the three-way cost policy

nb 13's threshold economics, upgraded to two dials. Blocking a legit sale costs margin +
goodwill (~$15); a review costs analyst time (~$2) but catches fraud it inspects;
allowing fraud costs the amount. And the review queue has a *capacity*:

In [ ]:
amounts_live = X[~split][:, 0]
y_live = y[~split]

def policy_cost(t_review: float, t_block: float):
    block = p_live >= t_block
    review = (p_live >= t_review) & ~block
    allow = ~block & ~review
    cost = (
        15.0 * (block & (y_live == 0)).sum()               # blocked good customers
        + 2.0 * review.sum()                               # analyst minutes
        + (amounts_live * (allow & (y_live == 1))).sum()   # fraud that walked through
    )                                                      # (reviewed fraud: caught)
    return cost, review.mean()

REVIEW_CAPACITY = 0.05
grid = np.linspace(0.02, 0.98, 33)
best = min(
    ((policy_cost(tr, tb), tr, tb)
     for tr in grid for tb in grid if tb > tr
     and policy_cost(tr, tb)[1] <= REVIEW_CAPACITY),
    key=lambda item: item[0][0],
)
(best_cost, review_load), t_review, t_block = best

cost_allow_all = (amounts_live * (y_live == 1)).sum()
cost_naive = policy_cost(0.5, 0.5)[0]                      # binary block at 0.5
print(f"chosen policy: review >= {t_review:.2f}, block >= {t_block:.2f} "
      f"(review load {review_load:.1%} of txns)")
print(f"cost per period - do nothing: ${cost_allow_all:,.0f}, "
      f"naive 0.5 block: ${cost_naive:,.0f}, tuned 3-way policy: ${best_cost:,.0f}")
assert best_cost < 0.6 * cost_allow_all, "the system pays for itself"
assert best_cost < cost_naive, "three-way beats binary: reviews are cheap insurance"
assert review_load <= REVIEW_CAPACITY, "the queue respects the analysts' actual capacity"

The thresholds are business dials on a config surface (nb 13): when fraud pressure or
analyst staffing changes, *the policy retunes without retraining* — and the review
queue's verdicts flow back as same-day proxy labels (nb 19's layered monitoring),
30 days ahead of the chargeback truth.

## Step 3d — Serving that degrades instead of dying

We are inside the checkout. nb 08 guarantees the feature store will sometimes time out
and the model will sometimes stall. The non-negotiable: **every request gets a decision
inside the budget.** Timeouts fall back — missing features to safe defaults + an
indicator (nb 16's missingness lesson), a dead model to a hard rule:

In [ ]:
def rule_fallback(amount: float) -> str:
    return "review" if amount > 120.0 else "allow"          # crude, but bounded loss

def decide(i: int, feature_timeout: bool, model_timeout: bool) -> tuple[str, str]:
    if model_timeout:
        return rule_fallback(X[~split][i, 0]), "rules"
    if feature_timeout:
        fallback_features = np.array([[X[~split][i, 0], 1.0, 0.0, 1.0, 86_400.0]])
        p = model.predict_proba(fallback_features)[0, 1]    # amount-only signal
        mode = "degraded"
    else:
        p = p_live[i]
        mode = "full"
    return ("block" if p >= t_block else "review" if p >= t_review else "allow"), mode

n = len(y_live)
feat_down = rng.random(n) < 0.02
model_down = rng.random(n) < 0.005
decisions, modes = zip(*(decide(i, feat_down[i], model_down[i]) for i in range(n)))
decisions, modes = np.array(decisions), np.array(modes)
print(f"decisions rendered: {n:,}/{n:,} (100.0%) - modes: "
      f"{dict(zip(*map(list, np.unique(modes, return_counts=True))))}")
assert len(decisions) == n, "no request is ever left hanging - the checkout survives us"

# Grade each path's quality COUNTERFACTUALLY over all live traffic (the random
# 2% outage slice holds too few frauds to measure anything, so measure the
# path, not the slice):
fraud = y_live == 1
X_degraded = np.c_[X[~split][:, 0], np.ones(n), np.zeros(n),
                   np.ones(n), np.full(n, 86_400.0)]
p_degraded = model.predict_proba(X_degraded)[:, 1]
recall_full = ((p_live >= t_review) & fraud).sum() / fraud.sum()
recall_degraded = ((p_degraded >= t_review) & fraud).sum() / fraud.sum()
recall_rules = ((X[~split][:, 0] > 120.0) & fraud).sum() / fraud.sum()
print(f"fraud flagged (block+review) if EVERY txn took the path - "
      f"full: {recall_full:.0%}, degraded features: {recall_degraded:.0%}, "
      f"rules only: {recall_rules:.0%}")
assert recall_full > 0.7
assert recall_full > max(recall_degraded, recall_rules), "fallbacks are worse - expected"
assert max(recall_degraded, recall_rules) > 0.2, \
    "and bounded: even the worst path catches real fraud - degradation is PLANNED, not free-fall"

Fallbacks are a designed feature contract, priced in advance: 2% of traffic at degraded
recall costs a computable expected loss — compare it to the revenue of never failing a
checkout, and the design defends itself in a sentence.

## Step 4 — Watching for the adversary (label delay, again)

Chargebacks confirm fraud ~30 days late; adversaries move in hours. The early-warning
stack from nb 19: **score-distribution PSI** (instant), review-queue hit rate (same
day), chargebacks (the truth, eventually). An attack wave, and the three clocks:

In [ ]:
def psi(ref, cur, bins=10):
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-6
    c = np.histogram(cur, edges)[0] / len(cur) + 1e-6
    return float(np.sum((c - r) * np.log(c / r)))

ATTACK_DAY, CHARGEBACK_DELAY = 12, 30
ref_scores = p_live[:4_000]
daily_psi, alarm_day = [], None
for day in range(20):
    day_scores = rng.choice(p_live, 800)                    # normal traffic sample
    if day >= ATTACK_DAY:                    # bot-driven attack wave: fraud x25
        frauds = p_live[y_live == 1]
        day_scores = np.concatenate([day_scores, rng.choice(frauds, 200)])
    score = psi(ref_scores, day_scores)
    daily_psi.append(score)
    if score > 0.1 and alarm_day is None:
        alarm_day = day

print(f"score-PSI alarm on day {alarm_day} (attack began day {ATTACK_DAY}); "
      f"chargebacks confirm around day {ATTACK_DAY + CHARGEBACK_DELAY}")
assert alarm_day == ATTACK_DAY, \
    "the score distribution betrayed the attack THE DAY it started - " \
    f"{CHARGEBACK_DELAY} days before ground truth could"

Plus the checks no dashboard should omit: slice metrics per merchant category (a flash
sale looks exactly like an attack — nb 17's slices prevent friendly fire), decision-mode
rates (rising "degraded" = infra trouble, not fraud trouble), and never echoing raw
scores to clients (adversaries binary-search your thresholds; return decisions, add
jitter near boundaries).

## Step 5 — Evolution & the standing failure analysis

Retraining rides nb 20–21's machinery (trigger on PSI + queue precision; gate on slice
metrics + calibration; shadow before canary). The permanent items on the risk register:
**selective labels** (blocked transactions never reveal outcomes — budgeted exploration
near the block threshold keeps training honest; nb 19 ex. 5, priced in dollars here);
**duplicate events** (a retried payment must not be double-scored into velocity features
— idempotency keys, nb 11); and **the model's own success** shifting the fraud that
remains (the adversarial version of nb 19's feedback loop: your training data is
whatever your defenses didn't stop).

## Exercises

1. **Per-amount thresholds.** A $9 transaction and a $2,000 transaction get the same
   `t_block` in our policy. Derive the per-transaction optimal block threshold from the
   cost matrix (nb 13's formula with amount as a variable) and state what curve
   replaces the single dial.
2. **Feature freshness triage.** Classify each feature (velocity 1h, amount-vs-habit,
   countries 24h, secs-since-prev): must it be updated synchronously *within* the
   request (before the decision), or is the async stream consumer (seconds of lag)
   safe? Construct the attack that exploits the wrong choice for velocity.
3. **The exploration budget.** You allow-through 0.5% of would-be-blocked transactions
   to keep labels unbiased. Price this budget (expected fraud loss) against the
   alternative (retraining on selectively-labeled data) — what measurable degradation
   in the *next* model justifies the spend?
4. **Idempotent money.** A network retry re-sends a transaction; between attempts, the
   model was updated and now says "block" where attempt 1 said "allow". Design the
   decision path (hint: decision cache keyed by the payment's idempotency key) and
   state which nb 11 guarantee you are implementing for *decisions*.
5. **Reason codes.** Regulation requires "why was this declined?" Name two feature-set
   design constraints this imposes, and how per-feature contributions from the GBM
   interact with the fallback-rules path (which must also explain itself).

### Solutions

**1.** Block iff `p × amount > 15 × (1 − p)` (fraud loss avoided vs expected good-sale
loss) ⇒ `p*(amount) = 15/(15 + amount)`: a $9 txn needs p > 0.63 to be worth blocking;
a $2,000 txn only p > 0.0074. The single dial becomes a *curve*; equivalently, rank by
expected loss `p × amount` — which is why real systems score in dollars, not
probabilities (nb 13's decoupling: the model stays in probability space, the policy
owns the money).

**2.** Velocity-1h and secs-since-prev must be **synchronous** — the burst *is* the
signal, and a stolen card fires 6 transactions in 5 minutes: with a lagging counter,
each one sees "0 recent transactions" and sails through; the attack is literally *rate
> your feature lag*. Amount-vs-habit and countries-24h move on human timescales —
seconds of stream lag are harmless (async, nb 16 ex. 5's tiering). The synchronous
update is why the decision service writes to the log *and* bumps the hot counters in
one request path.

**3.** Budget cost ≈ 0.5% × block-volume × E[amount | blocked] × P(fraud | blocked) —
concrete dollars per month. It buys: unbiased labels near the boundary, hence
calibrated `p` there (nb 17 ex. 2), hence correctly-placed thresholds. Justification
test: train a challenger *without* the exploration slice and compare boundary
calibration + tuned-policy cost (§3c) — when the no-exploration model's policy cost
exceeds the budget, the budget is cheap. (This is nb 20's measurement discipline
applied to a data-collection decision.)

**4.** First decision is computed, then **cached under the payment idempotency key**;
retries return the cached decision regardless of model updates — the decision, like the
charge, must be **effectively-once** (nb 11 §2, applied to an *output*). Consistency
beats freshness here: two different answers to one payment is how you charge and block
the same customer. Cache TTL ≥ retry horizon; the cache is itself in the fast path, so
it lives in the online store tier.

**5.** Constraints: features must be *individually meaningful* to a human ("6
transactions in the last hour", not `pca_7`), and mostly *monotone* in risk (nb 17's
monotonic constraints double as explanation hygiene — "risk rose because velocity
rose" must be true). GBM per-feature contributions map to reason codes; the fallback
rules are self-explaining by construction — which is an underrated argument for keeping
them simple, and for never letting the fallback path share the model's opacity.

## Takeaways — and the course, closed

- Point-in-time feature discipline is worth real money: the leaky builder **looked
  identical offline and crashed in production** — you measured both halves, and the
  offline invisibility is the whole danger. One definition, replayed for training,
  computed live for serving.
- Rare events + asymmetric costs ⇒ the deliverable is a **policy** (two dials, a
  capacity constraint, dollar objective), not a classifier. Retune without retraining.
- In the checkout path, availability outranks accuracy: designed fallbacks with
  bounded, priced degradation. Decisions, like charges, must be idempotent.
- Label delay is structural; layer the clocks: score-PSI (instant) → review verdicts
  (daily, biased) → chargebacks (true, late). Monitor slices or you'll shoot a flash
  sale.
- **The course in one paragraph:** keep an immutable ordered log of facts; derive every
  convenient shape from it; coordinate only where invariants demand; measure with
  percentiles and estimate before you build; make every learned component honest about
  time, evaluated past the average, deployed behind gates, watched for drift, and
  retrained as routine. Systems fail; design so that failing is a feature with a
  budget, not a surprise with a pager.

**Further reading.** Both books' final chapters, again — they read differently now.
Then: build something real, break it, and keep the log.

*— End of the course. The exercises are still the best part; go do the ones you
skipped.*